## Phase 2: Data Cleaning & Feature Selection (Part A)

This notebook prepares the KOI cumulative table for classification modeling

**Framing**: Part A trains on rows with a known ground-truth label - CONFIRMED AND FALSE POSITIVE - the applies the trained model to predict on CANDIDATE

In [1]:
import pandas as pd
import numpy as np

koi = pd.read_csv("../data/raw/koi_cumulative.csv", comment="#")
print("Loaded shape:", koi.shape)

Loaded shape: (9564, 153)


## Step 1: Drop Fully-Empty and Non-Predictive Columns

Dropping columns that are either completly empty across all rows, or non-numeric text fields that a tree-based/DL model can't use directly

In [2]:
fully_empty_cols = [
    "koi_ingress", "koi_model_chisq", "koi_sage", "koi_longp", "koi_model_dof",
    "koi_ingress_err1", "koi_ingress_err2", "koi_sage_err1", "koi_sage_err2",
    "koi_longp_err1", "koi_longp_err2", "koi_teq_err1", "koi_teq_err2",
    "koi_gmag_err", "koi_imag_err", "koi_zmag_err", "koi_rmag_err", "koi_kepmag_err",
    "koi_sma_err1", "koi_sma_err2", "koi_incl_err1", "koi_incl_err2",
    "koi_eccen_err1", "koi_eccen_err2"
]

non_predictive_cols = [
    "koi_comment", "koi_bin_oedp_sig",
    "kepoi_name", "kepler_name", "koi_delivname", "koi_vet_stat", "koi_vet_date",
    "koi_disp_prov", "koi_parm_prov", "koi_tce_delivname", "koi_datalink_dvs",
    "koi_datalink_dvr", "ra_str", "dec_str", "koi_sparprov", "koi_fittype",
    "koi_limbdark_mod", "koi_trans_mod"
]

drop_cols_stage1 = [c for c in fully_empty_cols + non_predictive_cols if c in koi.columns]
koi_clean = koi.drop(columns=drop_cols_stage1)

print(f"Dropped {len(drop_cols_stage1)} columns")
print("Remaining shape:", koi_clean.shape)

Dropped 42 columns
Remaining shape: (9564, 111)


### Step 2: Separate ID, Label, and Feature Columns

`kepid` is a pure identifier - useful for grouping (group-aware CV later) but not a model feature. `koi_disposition` and `koi_pdisposition` are labels.

`koi_score` is itself a NASA-vetting-pipeline output, similar in spirit to `koi_fpflag_*` - kept for now, but flagged for the same "does the model just learn to agree with the vetting pipeline" scrutiny as my original Kepler project applied

In [3]:
id_cols = ["kepid"]
label_cols = ["koi_disposition", "koi_pdisposition"]

feature_cols = [c for c in koi_clean.columns if c not in id_cols + label_cols]
print(f"ID Columns: {id_cols}")
print(f"Label Columns: {label_cols}")
print(f"Feature columns: {len(feature_cols)}")

ID Columns: ['kepid']
Label Columns: ['koi_disposition', 'koi_pdisposition']
Feature columns: 108


### Step 3: Handling Remaining Missing Values

Two groups:
1. Columns with substantial but non-total missingness carrying real signal (transit quality metrics, centroid/flux-weighted-motion columns) - median imputation
2. Any remaining sparse columns not yet addressed

In [4]:
impute_median_cols = [
    "koi_max_sngle_ev", "koi_quarters", "koi_num_transits", "koi_max_mult_ev",
    "koi_fwm_stat_sig", "koi_fwm_prao", "koi_fwm_pdeco",
    "koi_dicco_mdec", "koi_dicco_mra", "koi_dicco_msky",
    "koi_dikco_msky", "koi_dikco_mdec", "koi_zmag"
]
impute_median_cols = [c for c in impute_median_cols if c in koi_clean.columns]

for c in impute_median_cols:
    median_val = koi_clean[c].median()
    koi_clean[c] = koi_clean[c].fillna(median_val)

print(f"Median-imputed {len(impute_median_cols)} columns")

remaining_nulls = koi_clean[feature_cols].isnull().sum()
remaining_nulls = remaining_nulls[remaining_nulls > 0].sort_values(ascending=False)
print("\nColumns still containing nulls")
print(remaining_nulls)

Median-imputed 13 columns

Columns still containing nulls
koi_score             1510
koi_fwm_prao_err       830
koi_fwm_pdeco_err      817
koi_dicco_mra_err      599
koi_dicco_mdec_err     599
                      ... 
koi_jmag                25
koi_kmag                25
koi_hmag                25
koi_rmag                 9
koi_kepmag               1
Length: 82, dtype: int64


### Step 4: Handle Any Remaining Sparse Numeric Columns

Catch-all for anything Step 3 missed - impute numeric columns with median, and confirm zero nulls remain in the final feature set.

In [5]:
numeric_feature_cols = koi_clean[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

still_all_null = []
for col in numeric_feature_cols:
    if koi_clean[col].isnull().all():
        still_all_null.append(col)
    elif koi_clean[col].isnull().sum() > 0:
        koi_clean[col] = koi_clean[col].fillna(koi_clean[col].median())

if still_all_null:
    print("Dropping fully-null columns missed earlier:", still_all_null)
    koi_clean = koi_clean.drop(columns=still_all_null)
    feature_cols = [c for c in feature_cols if c not in still_all_null]

final_nulls = koi_clean[feature_cols].isnull().sum().sum()
print(f"Total remaining nulls across all feature columns: {final_nulls}")

Total remaining nulls across all feature columns: 0


### Step 5: Check for non numeric Columns Left in the feature set

Any remaining categorical/text columns need encoding or dropping before modeling

In [6]:
non_numeric_features = koi_clean[feature_cols].select_dtypes(exclude=[np.number]).columns.tolist()
print("Non-numeric feature columns remaining: ", non_numeric_features)
koi_clean[non_numeric_features].head() if non_numeric_features else print("None: all features are numeric")

Non-numeric feature columns remaining:  ['koi_quarters']


,koi_quarters
0,11111111111111111000000000000000
1,11111111111111111000000000000000
2,11111101110111011000000000000000
3,11111110111011101000000000000000
4,1111111111111111000000000000000


### Step 6: Prepare Train/Reference Split

Split into:
- `koi_labeled` - rows with CONFIRMED or FALSE POSITIVE (used for training/testing the classifier)
- `koi_candidates` - CANDIDATE rows (held out; the model predicts on these after training)

In [7]:
koi_labeled = koi_clean[koi_clean["koi_disposition"].isin(["CONFIRMED", "FALSE POSITIVE"])].copy()
koi_candidates = koi_clean[koi_clean["koi_disposition"] == "CANDIDATE"].copy()

koi_labeled["target"] = (koi_labeled["koi_disposition"] == "CONFIRMED").astype(int)

print("Labeled set shape:", koi_labeled.shape)
print("Candidate set shape:", koi_candidates.shape)
print("\nTarget balance in labeled set:")
print(koi_labeled["target"].value_counts())

Labeled set shape: (7587, 112)
Candidate set shape: (1977, 111)

Target balance in labeled set:
target
0    4839
1    2748
Name: count, dtype: int64


### Step 7: Save Processed Data

Saving cleaned, feature-ready datasets to `data/processed/` so Phase 3 (modeling) loads directly without repeating this pipeline.

In [8]:
import os
os.makedirs("../data/processed", exist_ok=True)

koi_labeled.to_csv("../data/processed/koi_labeled.csv", index=False)
koi_candidates.to_csv("../data/processed/koi_candidates.csv", index=False)

print("Saved koi_labeled.csv", koi_labeled.shape)
print("Saved koi_candidates.csv", koi_candidates.shape)

Saved koi_labeled.csv (7587, 112)
Saved koi_candidates.csv (1977, 111)


### Next Steps

Phase 3 will cover model training on `koi_labeled` (Random Forest, XGBoost, Gradient Boosting, SVM, Logistic Regression, and one DL model), using group-aware cross-validation (grouped by `kepid` where applicable) to avoid the host-star leakage issue identified in the earlier Kepler Project.